In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
x=pd.read_csv(r"C:\Users\shubh\OneDrive\Desktop\Sem-3\MSME\archive\news_dataset.csv")
x

,label,text
0,REAL,Payal has accused filmmaker Anurag Kashyap of ...
1,FAKE,A four-minute-long video of a woman criticisin...
2,FAKE,"Republic Poll, a fake Twitter account imitatin..."
3,REAL,"Delhi teen finds place on UN green list, turns..."
4,REAL,Delhi: A high-level meeting underway at reside...
...,...,...
3724,REAL,19:17 (IST) Sep 20\n\nThe second round of coun...
3725,REAL,19:17 (IST) Sep 20\n\nThe second round of coun...
3726,FAKE,The Bengaluru City Police’s official Twitter h...
3727,REAL,"Sep 20, 2020, 08:00AM IST\n\nSource: TOI.in\n\..."


In [5]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
x.isnull().sum()

label    0
text     8
dtype: int64

In [9]:
x=x.dropna()

In [19]:
x.label.value_counts()

label
0    1871
1    1850
Name: count, dtype: int64

In [15]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

In [17]:
x.label=le.fit_transform(x.label)
x.label.unique()

array([1, 0])

In [21]:
ip=x.drop('label',axis=1)
ip.head()

,text
0,Payal has accused filmmaker Anurag Kashyap of ...
1,A four-minute-long video of a woman criticisin...
2,"Republic Poll, a fake Twitter account imitatin..."
3,"Delhi teen finds place on UN green list, turns..."
4,Delhi: A high-level meeting underway at reside...


In [23]:
op=x.drop(ip,axis=1)
op.head()

,label
0,1
1,0
2,0
3,1
4,1


In [25]:
from sklearn.model_selection import train_test_split
xtr,xts,ytr,yts=train_test_split(ip,op,test_size=0.2,random_state=13)

In [27]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # remove URLs
    text = re.sub(r'\@w+|\#','', text)  # remove mentions and hashtags
    text = re.sub(r'[^A-Za-z\s]', '', text)  # remove punctuations
    text = text.split()
    text = [stemmer.stem(word) for word in text if word not in stop_words]
    return ' '.join(text)


In [29]:
x.text.isnull().sum()

0

In [31]:
xtr['text'] = xtr['text'].apply(clean_text)
xts['text'] = xts['text'].apply(clean_text)


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
xtr_vec = vectorizer.fit_transform(xtr['text']).toarray()
xts_vec = vectorizer.transform(xts['text']).toarray()


In [34]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score


In [36]:
# SVM
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(xtr_vec, ytr)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=13)
rf_model.fit(xtr_vec, ytr)

# XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(xtr_vec, ytr)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [38]:
models = {
    'SVM': svm_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

for name, model in models.items():
    print(f"--- {name} ---")
    preds = model.predict(xts_vec)
    print("Accuracy:", accuracy_score(yts, preds))
    print("Classification Report:\n", classification_report(yts, preds))
    print()


--- SVM ---
Accuracy: 0.9959731543624161
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       370
           1       0.99      1.00      1.00       375

    accuracy                           1.00       745
   macro avg       1.00      1.00      1.00       745
weighted avg       1.00      1.00      1.00       745


--- Random Forest ---
Accuracy: 0.9959731543624161
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       370
           1       1.00      0.99      1.00       375

    accuracy                           1.00       745
   macro avg       1.00      1.00      1.00       745
weighted avg       1.00      1.00      1.00       745


--- XGBoost ---
Accuracy: 0.9973154362416108
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       370
           1       1.00   

In [40]:
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam


In [35]:
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam

model = Sequential()
model.add(Dense(128, input_dim=xtr_vec.shape[1], activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

history = model.fit(xtr_vec, ytr, epochs=3, batch_size=32, validation_data=(xts_vec, yts))


Epoch 1/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8850 - loss: 0.4719 - val_accuracy: 0.9960 - val_loss: 0.0236
Epoch 2/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9992 - loss: 0.0137 - val_accuracy: 0.9946 - val_loss: 0.0119
Epoch 3/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.9960 - val_loss: 0.0105


In [37]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

y_pred_dl = model.predict(xts_vec)
y_pred_labels = (y_pred_dl > 0.5).astype(int)

print("Accuracy:", accuracy_score(yts, y_pred_labels))
print("Recall:", recall_score(yts, y_pred_labels))
print("Precision:", precision_score(yts, y_pred_labels))
print("F1 Score:", f1_score(yts, y_pred_labels))


24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Accuracy: 0.9959731543624161
Recall: 0.9946666666666667
Precision: 0.9973262032085561
F1 Score: 0.9959946595460614


In [50]:
def predict_news_article(text, model, vectorizer):
    import re
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer

    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()

    def clean_text(t):
        t = t.lower()
        t = re.sub(r"http\S+|www\S+|https\S+", '', t)
        t = re.sub(r'\@w+|\#', '', t)
        t = re.sub(r'[^a-zA-Z\s]', '', t)
        t = t.split()
        t = [stemmer.stem(word) for word in t if word not in stop_words]
        return ' '.join(t)

    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned]).toarray()
    pred = model.predict(vec)[0]

    return "REAL" if pred == 1 else "FAKE"


In [106]:
article = """Senate Majority Whip John Cornyn (R-TX) thought it would be a good idea to attack Special Counsel Robert Mueller over the Russia probe. As Mueller s noose tightens, Republicans are losing their sh-t and attacking Mueller and the FBI in order to protect probably the most corrupt  president  ever.Former Attorney General Eric Holder tweeted on Friday,  Speaking on behalf of the vast majority of the American people, Republicans in Congress be forewarned: any attempt to remove Bob Mueller will not be tolerated. Cornyn retweeted Holder to say,  You don t. You don t https://t.co/7lHYkIloyz  Senator JohnCornyn (@JohnCornyn) December 16, 2017Bloomberg s Steven Dennis tweeted on Saturday that  [Cornyn] s beef is with Holder, not Mueller,  but Cornyn responded to say,  But Mueller needs to clean house of partisans. But Mueller needs to clean house of partisans https://t.co/g8SwgAKtfH  Senator JohnCornyn (@JohnCornyn) December 16, 2017The Washington Post s Greg Sargent asked Cornyn,  Will you accept the findings of the Mueller probe as legitimate, @JohnCornyn? Makes sense to me to wait to see what they are first,  Cornyn responded.Makes sense to me to wait to see what they are first https://t.co/9lCqpYujKN  Senator JohnCornyn (@JohnCornyn) December 16, 2017Republicans are trying to discredit Mueller and Twitter users took notice.If you even THINK of firing Mueller I ll make it my life s mission to make sure this is your last term, buddy.  Mrs. SMH (@MRSSMH2) December 16, 2017Carrollton, TX here Ready and willing to help get Cruz and Cornyn out  Jules012 (@JulesLorey1) December 16, 2017Garland, TX here Same! Bye Bye   TDK (@ejkmom1998) December 16, 2017Austin, TX. #IStandWithMueller. Cronyn is a fake representative. He represents his own interests and anything to profit himself  Vj (@Tex92eye) December 16, 2017I stand with Mueller!   Kenneth Shipp (@shipp_kenneth) December 16, 2017He speaks for me Its BS how you cover up for a Russia pawn If Trump not gulity why would Mueller be fired to cover up  Ellen Reeher Morris (@EllenMorris1222) December 16, 2017@EricHolder speaks for 69% of Americans according to recent polling. That s  vast majority  in my book. You were around for the Saturday Night Massacre @JohnCornyn. Firing Mueller would be X100!  Lori Winters (@LoriW66) December 16, 2017Country over party. pic.twitter.com/NXEX9rGBgu  PittieBoo (@PittieBoo) December 16, 2017He speaks for me, @JohnCornyn , and he speaks for the vast majority of American citizens who, you should remember, vote. See you in 2018.  Andrew Silver (@standsagreenoak) December 16, 2017I might just move to Texas to get those cronies tossed out of office. Blue wave is coming for the corrupt.  Ollie (@marciebp) December 16, 2017Good try, John. History will not be kind to you.Photo by Ann Heisenfelt/Getty Images
"""
result = predict_news_article(article, rf_model, vectorizer)
print("Prediction:", result)


Prediction: FAKE


In [94]:
article = """NEW YORK (Reuters) - The U.S. Justice Department has issued new guidelines for immigration judges that remove some instructions for how to protect unaccompanied juveniles appearing in their courtrooms. A Dec. 20 memo, issued by the Executive Office for Immigration Review (EOIR) replaces 2007 guidelines, spelling out policies and procedures judges should follow in dealing with children who crossed the border illegally alone and face possible deportation.  The new memo removes suggestions contained in the 2007 memo for how to conduct â€œchild-sensitive questioningâ€ and adds reminders to judges to maintain â€œimpartialityâ€ even though â€œjuvenile cases may present sympathetic allegations.â€ The new document also changes the word â€œchildâ€ to â€œunmarried individual under the age of 18â€ in many instances. (Link to comparison: tmsnrt.rs/2BlT0VK May 2007 document: tmsnrt.rs/2BBR8wj December 2017 document: tmsnrt.rs/2C2sWCs)  An EOIR official said the new memo contained â€œclarifications and updatesâ€ to 10-year-old guidance â€œin order to be consistent with the laws as theyâ€™ve been passed by Congress.â€ The new memo was posted on the Justice Department website but has not been previously reported.  Immigration advocates said they worry the new guidelines could make court appearances for children more difficult, and a spokeswoman for the union representing immigration judges said judges are concerned about the tone of the memo. President Donald Trump has made tougher immigration enforcement a key policy goal of his administration, and has focused particularly on trying to curb the illegal entry of children. The administration says it wants to prevent vulnerable juveniles from making perilous journeys to the United States and eliminate fraud from programs for young immigrants.  One changed section of the memo focuses on how to make children comfortable in the court in advance of hearings. The old guidance says they â€œshould be permitted to exploreâ€ courtrooms and allowed to â€œsit in all locations, (including, especially, the judgeâ€™s bench and the witness stand).â€  The new guidance says such explorations should take place only â€œto the extent that resources and time permitâ€ and specifically puts the judgeâ€™s bench off limits. The new memo also warns judges to be skeptical, since an unaccompanied minor â€œgenerally receives more favorable treatment under the law than other categories of illegal aliens,â€ which creates â€œan incentive to misrepresent accompaniment status or age in order to attempt to qualify for the benefits.â€ It also says to be on the lookout for â€œfraud and abuse,â€ language that was not in the previous memo. Immigration judges are appointed by the U.S. Attorney General and courts are part of the Department of Justice, not an independent branch. The only sitting immigration judges routinely allowed to speak to the media are representatives of their union, the National Association of Immigration Judges.  Dana Marks, a sitting judge and spokeswoman for the union, said the â€œoverall toneâ€ of the memo â€œis very distressing and concerning to immigration judges.â€  â€œThere is a feeling that the immigration courts are just being demoted into immigration enforcement offices, rather than neutral arbiters,â€ Marks said. â€œThere has been a relentless beating of the drum toward enforcement rather than due process.â€   Former immigration judge Andrew Arthur, who now works at the Center for Immigration Studies, which promotes lower levels of immigration overall, said the new guidelines were needed.  In their previous form, he said, â€œso much emphasis was placed on the potential inability of the alien to understand the proceedings ... that it almost put the judge into the position of being an advocate.â€Â   The courts have had to handle a surge in cases for unaccompanied minors, mostly from Central America, after their numbers sky-rocketed in 2014 as violence in the region caused residents to flee north.  While illegal crossings initially fell after Trump took office, U.S. Customs and Border Protection said that since May, each month has seen an increase in children being apprehended either alone or with family members.  Attorney General Jeff Sessions said in a speech in Boston in September that the special accommodations for unaccompanied minors had been exploited by â€œgang members who come to this country as wolves in sheep clothing.â€ Echoing some of these concerns, the new memo notes in a preamble that not all child cases involve innocents, and that the courts might see â€œan adolescent gang memberâ€ or â€œa teenager convicted as an adult for serious criminal activity.â€  Jennifer Podkul, policy director of Kids in Need of Defense (KIND) said Congress included special procedural protections for immigrant children in a 2008 anti-trafficking bill to â€œmake sure that a kid gets a fair shot in the courtroom.â€ â€œThese kids are by themselves telling a very complicated and oftentimes very traumatic story,â€ said Podkul. â€œThe approach of this memo, which is much more suspicious, is not going to help get to the truth of a childâ€™s story.â€  In cases where children are called to testify, the old guidance instructed judges to â€œseek to limit the amount of time the child is on the stand.â€ The new guidance says that judges should â€œconsiderâ€ limiting the childâ€™s time on the stand â€œwithout compromising due process for the opposing party,â€ which is generally a government prosecutor. The memo leaves in a range of special accommodations made for children, including allowing them to bring a pillow or booster seat or a â€œtoy, book, or other personal item.â€ It also maintains that cases involving unaccompanied minors should be heard on a separate docket when possible and that children should not be detained or transported with adults. 
"""
result = predict_news_article(article, model, vectorizer)
print("Prediction:", result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction: FAKE


In [104]:
article = """Senate Majority Whip John Cornyn (R-TX) thought it would be a good idea to attack Special Counsel Robert Mueller over the Russia probe. As Mueller s noose tightens, Republicans are losing their sh-t and attacking Mueller and the FBI in order to protect probably the most corrupt  president  ever.Former Attorney General Eric Holder tweeted on Friday,  Speaking on behalf of the vast majority of the American people, Republicans in Congress be forewarned: any attempt to remove Bob Mueller will not be tolerated. Cornyn retweeted Holder to say,  You don t. You don t https://t.co/7lHYkIloyz  Senator JohnCornyn (@JohnCornyn) December 16, 2017Bloomberg s Steven Dennis tweeted on Saturday that  [Cornyn] s beef is with Holder, not Mueller,  but Cornyn responded to say,  But Mueller needs to clean house of partisans. But Mueller needs to clean house of partisans https://t.co/g8SwgAKtfH  Senator JohnCornyn (@JohnCornyn) December 16, 2017The Washington Post s Greg Sargent asked Cornyn,  Will you accept the findings of the Mueller probe as legitimate, @JohnCornyn? Makes sense to me to wait to see what they are first,  Cornyn responded.Makes sense to me to wait to see what they are first https://t.co/9lCqpYujKN  Senator JohnCornyn (@JohnCornyn) December 16, 2017Republicans are trying to discredit Mueller and Twitter users took notice.If you even THINK of firing Mueller I ll make it my life s mission to make sure this is your last term, buddy.  Mrs. SMH (@MRSSMH2) December 16, 2017Carrollton, TX here Ready and willing to help get Cruz and Cornyn out  Jules012 (@JulesLorey1) December 16, 2017Garland, TX here Same! Bye Bye   TDK (@ejkmom1998) December 16, 2017Austin, TX. #IStandWithMueller. Cronyn is a fake representative. He represents his own interests and anything to profit himself  Vj (@Tex92eye) December 16, 2017I stand with Mueller!   Kenneth Shipp (@shipp_kenneth) December 16, 2017He speaks for me Its BS how you cover up for a Russia pawn If Trump not gulity why would Mueller be fired to cover up  Ellen Reeher Morris (@EllenMorris1222) December 16, 2017@EricHolder speaks for 69% of Americans according to recent polling. That s  vast majority  in my book. You were around for the Saturday Night Massacre @JohnCornyn. Firing Mueller would be X100!  Lori Winters (@LoriW66) December 16, 2017Country over party. pic.twitter.com/NXEX9rGBgu  PittieBoo (@PittieBoo) December 16, 2017He speaks for me, @JohnCornyn , and he speaks for the vast majority of American citizens who, you should remember, vote. See you in 2018.  Andrew Silver (@standsagreenoak) December 16, 2017I might just move to Texas to get those cronies tossed out of office. Blue wave is coming for the corrupt.  Ollie (@marciebp) December 16, 2017Good try, John. History will not be kind to you.Photo by Ann Heisenfelt/Getty Images
"""
result = predict_news_article(article, xgb_model, vectorizer)
print("Prediction:", result)


Prediction: FAKE


In [102]:
article = """Senate Majority Whip John Cornyn (R-TX) thought it would be a good idea to attack Special Counsel Robert Mueller over the Russia probe. As Mueller s noose tightens, Republicans are losing their sh-t and attacking Mueller and the FBI in order to protect probably the most corrupt  president  ever.Former Attorney General Eric Holder tweeted on Friday,  Speaking on behalf of the vast majority of the American people, Republicans in Congress be forewarned: any attempt to remove Bob Mueller will not be tolerated. Cornyn retweeted Holder to say,  You don t. You don t https://t.co/7lHYkIloyz  Senator JohnCornyn (@JohnCornyn) December 16, 2017Bloomberg s Steven Dennis tweeted on Saturday that  [Cornyn] s beef is with Holder, not Mueller,  but Cornyn responded to say,  But Mueller needs to clean house of partisans. But Mueller needs to clean house of partisans https://t.co/g8SwgAKtfH  Senator JohnCornyn (@JohnCornyn) December 16, 2017The Washington Post s Greg Sargent asked Cornyn,  Will you accept the findings of the Mueller probe as legitimate, @JohnCornyn? Makes sense to me to wait to see what they are first,  Cornyn responded.Makes sense to me to wait to see what they are first https://t.co/9lCqpYujKN  Senator JohnCornyn (@JohnCornyn) December 16, 2017Republicans are trying to discredit Mueller and Twitter users took notice.If you even THINK of firing Mueller I ll make it my life s mission to make sure this is your last term, buddy.  Mrs. SMH (@MRSSMH2) December 16, 2017Carrollton, TX here Ready and willing to help get Cruz and Cornyn out  Jules012 (@JulesLorey1) December 16, 2017Garland, TX here Same! Bye Bye   TDK (@ejkmom1998) December 16, 2017Austin, TX. #IStandWithMueller. Cronyn is a fake representative. He represents his own interests and anything to profit himself  Vj (@Tex92eye) December 16, 2017I stand with Mueller!   Kenneth Shipp (@shipp_kenneth) December 16, 2017He speaks for me Its BS how you cover up for a Russia pawn If Trump not gulity why would Mueller be fired to cover up  Ellen Reeher Morris (@EllenMorris1222) December 16, 2017@EricHolder speaks for 69% of Americans according to recent polling. That s  vast majority  in my book. You were around for the Saturday Night Massacre @JohnCornyn. Firing Mueller would be X100!  Lori Winters (@LoriW66) December 16, 2017Country over party. pic.twitter.com/NXEX9rGBgu  PittieBoo (@PittieBoo) December 16, 2017He speaks for me, @JohnCornyn , and he speaks for the vast majority of American citizens who, you should remember, vote. See you in 2018.  Andrew Silver (@standsagreenoak) December 16, 2017I might just move to Texas to get those cronies tossed out of office. Blue wave is coming for the corrupt.  Ollie (@marciebp) December 16, 2017Good try, John. History will not be kind to you.Photo by Ann Heisenfelt/Getty Images
"""
result = predict_news_article(article, svm_model, vectorizer)
print("Prediction:", result)


Prediction: REAL


In [108]:
import joblib

# Save the trained XGBoost model
#joblib.dump(xgb_model, "xgb_model.pkl")


['xgb_model.pkl']

In [112]:
# Save the TF-IDF vectorizer used for preprocessing
#joblib.dump(vectorizer, "tfidf_vectorizer_xgb.pkl")


['tfidf_vectorizer_xgb.pkl']

In [ ]:
# Load model
#xgb_model = joblib.load("xgb_model.pkl")

# Load vectorizer
#vectorizer = joblib.load("tfidf_vectorizer.pkl")


In [39]:
#model.save("news_model.h5")

In [41]:
import pickle

#with open("vectorizer_news.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
